<a href="https://colab.research.google.com/github/apichaia/Mini-Project/blob/dbt_duckdb/MINIPROJECT_CODE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install duckdb -q

import pandas as pd
import duckdb
import os

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
path = '/content/drive/My Drive/miniproject_สินค้าปลีก'

print("ตรวจสอบโฟลเดอร์ Dataset")
print(os.listdir(path))

ตรวจสอบโฟลเดอร์ Dataset
['promotions.csv', 'payments.csv', 'customers.csv', 'products.csv', 'returns.csv', 'stores.csv', 'suppliers.csv', 'shipments.csv', 'employees.csv', 'categories.csv', 'orders.csv', 'order_items.csv', 'data_01.txt', 'archive_02.zip', 'retail.duckdb']


In [5]:
import os
import pandas as pd

tables = [
    "employees",
    "returns",
    "products",
    "suppliers",
    "categories",
    "promotions",
    "stores",
    "customers",
    "payments",
    "orders",
    "order_items",
    "shipments"
]

loaded_data = {}

for table in tables:
    file_path = os.path.join(path, table + ".csv")

    df = pd.read_csv(file_path)

    loaded_data[table] = df

    print(
        f"{table}.csv -> "
        f"{len(df):,} records, "
        f"{len(df.columns)} columns"
    )

employees.csv -> 1,000 records, 3 columns
returns.csv -> 30,000 records, 3 columns
products.csv -> 10,000 records, 4 columns
suppliers.csv -> 200 records, 2 columns
categories.csv -> 30 records, 2 columns
promotions.csv -> 50 records, 2 columns
stores.csv -> 100 records, 2 columns
customers.csv -> 50,000 records, 3 columns
payments.csv -> 300,000 records, 3 columns
orders.csv -> 300,000 records, 5 columns
order_items.csv -> 600,000 records, 5 columns
shipments.csv -> 300,000 records, 3 columns


In [6]:
db_path = os.path.join(path, 'retail.duckdb')

con = duckdb.connect(db_path)

print("เชื่อมต่อ DuckDB สำเร็จ")
print("Database:", db_path)

เชื่อมต่อ DuckDB สำเร็จ
Database: /content/drive/My Drive/miniproject_สินค้าปลีก/retail.duckdb


In [7]:
import duckdb

con = duckdb.connect("retail_dw.duckdb")

for table in tables:

    con.execute(f"""
        CREATE OR REPLACE TABLE stg_{table} AS
        SELECT *
        FROM read_csv_auto(?)
    """, [os.path.join(path, table + ".csv")])

    print(f"Created staging table: stg_{table}")

Created staging table: stg_employees
Created staging table: stg_returns
Created staging table: stg_products
Created staging table: stg_suppliers
Created staging table: stg_categories
Created staging table: stg_promotions
Created staging table: stg_stores
Created staging table: stg_customers
Created staging table: stg_payments
Created staging table: stg_orders
Created staging table: stg_order_items
Created staging table: stg_shipments


In [10]:
tables_in_db = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").df()

tables_in_db

,table_name
0,dim_customer
1,dim_date
2,dim_employee
3,dim_product
4,dim_promotion
5,dim_store
6,dim_supplier
7,fact_payments
8,fact_return
9,fact_sales


In [11]:
load_result = []

for table in tables:

    count = con.execute(
        f"SELECT COUNT(*) FROM stg_{table}"
    ).fetchone()[0]

    load_result.append({
        'Source File': table + '.csv',
        'Staging Table': 'stg_' + table,
        'Records': count
    })

load_summary = pd.DataFrame(load_result)

load_summary

,Source File,Staging Table,Records
0,employees.csv,stg_employees,1000
1,returns.csv,stg_returns,30000
2,products.csv,stg_products,10000
3,suppliers.csv,stg_suppliers,200
4,categories.csv,stg_categories,30
5,promotions.csv,stg_promotions,50
6,stores.csv,stg_stores,100
7,customers.csv,stg_customers,50000
8,payments.csv,stg_payments,300000
9,orders.csv,stg_orders,300000


In [12]:
total_records = load_summary['Records'].sum()

print(f"จำนวนตารางทั้งหมด : {len(load_summary)} ตาราง")
print(f"จำนวน Records รวม : {total_records:,} Records")

จำนวนตารางทั้งหมด : 12 ตาราง
จำนวน Records รวม : 1,591,380 Records


In [13]:
for table in tables:

    print("=" * 60)
    print(f"TABLE: stg_{table}")
    print("=" * 60)

    result = con.execute(
        f"DESCRIBE stg_{table}"
    ).df()

    display(result)

TABLE: stg_employees


,column_name,column_type,null,key,default,extra
0,employee_id,BIGINT,YES,None,None,None
1,store_id,BIGINT,YES,None,None,None
2,salary,BIGINT,YES,None,None,None


TABLE: stg_returns


,column_name,column_type,null,key,default,extra
0,return_id,BIGINT,YES,None,None,None
1,order_item_id,BIGINT,YES,None,None,None
2,refund,BIGINT,YES,None,None,None


TABLE: stg_products


...

,column_name,column_type,null,key,default,extra
0,order_id,BIGINT,YES,None,None,None
1,customer_id,BIGINT,YES,None,None,None
2,store_id,BIGINT,YES,None,None,None
3,order_date,DATE,YES,None,None,None
4,promotion_id,BIGINT,YES,None,None,None


TABLE: stg_order_items


,column_name,column_type,null,key,default,extra
0,order_item_id,BIGINT,YES,None,None,None
1,order_id,BIGINT,YES,None,None,None
2,product_id,BIGINT,YES,None,None,None
3,qty,BIGINT,YES,None,None,None
4,price,BIGINT,YES,None,None,None


TABLE: stg_shipments


,column_name,column_type,null,key,default,extra
0,shipment_id,BIGINT,YES,None,None,None
1,order_id,BIGINT,YES,None,None,None
2,status,VARCHAR,YES,None,None,None


In [14]:
for table in tables:

    print("=" * 60)
    print(f"Missing Value: stg_{table}")
    print("=" * 60)

    df = con.execute(
        f"SELECT * FROM stg_{table}"
    ).df()

    missing = df.isnull().sum()

    print(missing)

Missing Value: stg_employees
employee_id    0
store_id       0
salary         0
dtype: int64
Missing Value: stg_returns
return_id        0
order_item_id    0
refund           0
dtype: int64
Missing Value: stg_products
product_id     0
category_id    0
supplier_id    0
price          0
dtype: int64
Missing Value: stg_suppliers
supplier_id    0
country        0
dtype: int64
Missing Value: stg_categories
category_id      0
category_name    0
dtype: int64
Missing Value: stg_promotions
promotion_id    0
discount        0
dtype: int64
Missing Value: stg_stores
store_id    0
city        0
dtype: int64
Missing Value: stg_customers
customer_id    0
city           0
signup_date    0
dtype: int64
Missing Value: stg_payments
payment_id    0
order_id      0
amount        0
dtype: int64
Missing Value: stg_orders
order_id        0
customer_id     0
store_id        0
order_date      0
promotion_id    0
dtype: int64
Missing Value: stg_order_items
order_item_id    0
order_id         0
product_id       0

In [ ]:
for table in tables:

    duplicate_count = con.execute(f"""
        SELECT COUNT(*)
        FROM (
            SELECT *
            FROM stg_{table}
            GROUP BY ALL
            HAVING COUNT(*) > 1
        )
    """).fetchone()[0]

    print(f"{table:15} : {duplicate_count} duplicate groups")

In [15]:
primary_keys = {
    'employees': 'employee_id',
    'returns': 'return_id',
    'products': 'product_id',
    'suppliers': 'supplier_id',
    'categories': 'category_id',
    'promotions': 'promotion_id',
    'stores': 'store_id',
    'customers': 'customer_id',
    'payments': 'payment_id',
    'orders': 'order_id',
    'order_items': 'order_item_id',
    'shipments': 'shipment_id'
}

for table, pk in primary_keys.items():

    duplicate_pk = con.execute(f"""
        SELECT COUNT(*)
        FROM (
            SELECT {pk}
            FROM stg_{table}
            GROUP BY {pk}
            HAVING COUNT(*) > 1
        )
    """).fetchone()[0]

    print(f"{table:15} | PK = {pk:15} | Duplicate PK = {duplicate_pk}")

employees       | PK = employee_id     | Duplicate PK = 0
returns         | PK = return_id       | Duplicate PK = 0
products        | PK = product_id      | Duplicate PK = 0
suppliers       | PK = supplier_id     | Duplicate PK = 0
categories      | PK = category_id     | Duplicate PK = 0
promotions      | PK = promotion_id    | Duplicate PK = 0
stores          | PK = store_id        | Duplicate PK = 0
customers       | PK = customer_id     | Duplicate PK = 0
payments        | PK = payment_id      | Duplicate PK = 0
orders          | PK = order_id        | Duplicate PK = 0
order_items     | PK = order_item_id   | Duplicate PK = 0
shipments       | PK = shipment_id     | Duplicate PK = 0


In [16]:
con.execute("""
SELECT *
FROM stg_orders
LIMIT 10
""").df()

,order_id,customer_id,store_id,order_date,promotion_id
0,1,45308,33,2021-08-26,24
1,2,10070,81,2022-03-19,3
2,3,43308,17,2021-01-21,25
3,4,47997,85,2021-01-16,48
4,5,36546,81,2022-09-14,33
5,6,9479,84,2023-02-03,2
6,7,33431,77,2022-10-29,44
7,8,33130,61,2022-10-10,12
8,9,39339,29,2021-07-09,47
9,10,28094,21,2022-06-03,21


In [17]:
con.execute("""
SELECT *
FROM stg_order_items
LIMIT 10
""").df()

,order_item_id,order_id,product_id,qty,price
0,1,145042,472,3,176
1,2,110932,1666,2,1034
2,3,269799,8616,4,2290
3,4,298741,9909,3,1555
4,5,105218,1179,3,936
5,6,162991,6221,4,248
6,7,259284,1104,3,790
7,8,205732,8451,4,4701
8,9,184725,9946,2,1858
9,10,159121,5640,1,782


In [18]:
con.execute("""
SELECT
    o.order_id,
    o.customer_id,
    o.store_id,
    oi.product_id,
    oi.qty,
    oi.price
FROM stg_orders o
JOIN stg_order_items oi
    ON o.order_id = oi.order_id
LIMIT 10
""").df()

,order_id,customer_id,store_id,product_id,qty,price
0,28627,9644,52,4517,2,1749
1,134640,36,93,875,1,2759
2,122620,19899,29,8943,1,738
3,281696,37928,11,1363,3,1260
4,165593,38643,5,5214,2,3416
5,159051,21903,96,8785,4,1163
6,119976,44182,16,7642,1,3941
7,153553,5055,58,9791,2,3611
8,295748,3136,100,7089,4,4999
9,210042,18630,83,4224,1,4854


In [19]:
# ============================================
# 1.3.1 DIMENSION TABLES
# ============================================

# --------------------------------------------
# DIM DATE
# --------------------------------------------

con.execute("""
CREATE OR REPLACE TABLE dim_date AS
SELECT DISTINCT
    CAST(
        STRFTIME(
            TRY_CAST(order_date AS DATE),
            '%Y%m%d'
        ) AS INTEGER
    ) AS date_id,

    EXTRACT(
        YEAR FROM TRY_CAST(order_date AS DATE)
    ) AS year,

    EXTRACT(
        QUARTER FROM TRY_CAST(order_date AS DATE)
    ) AS quarter,

    EXTRACT(
        MONTH FROM TRY_CAST(order_date AS DATE)
    ) AS month,

    EXTRACT(
        DAY FROM TRY_CAST(order_date AS DATE)
    ) AS day

FROM stg_orders
WHERE TRY_CAST(order_date AS DATE) IS NOT NULL
ORDER BY date_id
""")


# --------------------------------------------
# DIM PRODUCT
# --------------------------------------------

con.execute("""
CREATE OR REPLACE TABLE dim_product AS
SELECT
    product_id,
    category_id,
    supplier_id,
    price
FROM stg_products
""")


# --------------------------------------------
# DIM CUSTOMER
# --------------------------------------------

con.execute("""
CREATE OR REPLACE TABLE dim_customer AS
SELECT
    customer_id,
    city,
    TRY_CAST(signup_date AS DATE) AS signup_date
FROM stg_customers
""")


# --------------------------------------------
# DIM STORE
# --------------------------------------------

con.execute("""
CREATE OR REPLACE TABLE dim_store AS
SELECT
    store_id,
    city
FROM stg_stores
""")


# --------------------------------------------
# DIM PROMOTION
# --------------------------------------------

con.execute("""
CREATE OR REPLACE TABLE dim_promotion AS
SELECT
    promotion_id,
    discount
FROM stg_promotions
""")


# --------------------------------------------
# DIM SUPPLIER
# --------------------------------------------

con.execute("""
CREATE OR REPLACE TABLE dim_supplier AS
SELECT
    supplier_id,
    country
FROM stg_suppliers
""")


# --------------------------------------------
# DIM EMPLOYEE
# --------------------------------------------

con.execute("""
CREATE OR REPLACE TABLE dim_employee AS
SELECT
    employee_id,
    store_id,
    salary
FROM stg_employees
""")


print("Dimension Tables created successfully.")


# ============================================
# 1.3.2 FACT TABLES
# ============================================

# --------------------------------------------
# FACT SALES
# Grain: 1 Order Line Item
# --------------------------------------------

con.execute("""
CREATE OR REPLACE TABLE fact_sales AS

SELECT
    oi.order_item_id AS order_items_id,

    CAST(
        STRFTIME(
            TRY_CAST(o.order_date AS DATE),
            '%Y%m%d'
        ) AS INTEGER
    ) AS date_id,

    oi.product_id,
    o.customer_id,
    o.store_id,
    o.promotion_id,
    p.supplier_id,
    oi.order_id,

    CAST(oi.qty AS INTEGER) AS quantity,

    CAST(oi.price AS NUMERIC) AS unit_price,

    CAST(
        oi.qty * oi.price
        AS NUMERIC
    ) AS sales_amount,

    CAST(
        COALESCE(pr.discount, 0)
        AS NUMERIC
    ) AS discount

FROM stg_order_items oi

LEFT JOIN stg_orders o
    ON oi.order_id = o.order_id

LEFT JOIN stg_products p
    ON oi.product_id = p.product_id

LEFT JOIN stg_promotions pr
    ON o.promotion_id = pr.promotion_id
""")


# --------------------------------------------
# FACT RETURN
# Grain: 1 Return Transaction
# --------------------------------------------

con.execute("""
CREATE OR REPLACE TABLE fact_return AS

SELECT
    r.return_id,

    CAST(
        STRFTIME(
            TRY_CAST(o.order_date AS DATE),
            '%Y%m%d'
        ) AS INTEGER
    ) AS date_id,

    oi.product_id,
    o.customer_id,
    o.store_id,

    r.order_item_id AS order_items_id,

    CAST(r.refund AS NUMERIC) AS refund

FROM stg_returns r

LEFT JOIN stg_order_items oi
    ON r.order_item_id = oi.order_item_id

LEFT JOIN stg_orders o
    ON oi.order_id = o.order_id
""")


Dimension Tables created successfully.


In [20]:
# --------------------------------------------
# FACT SHIPMENTS
# Grain: 1 Shipment
# --------------------------------------------

con.execute("""
CREATE OR REPLACE TABLE fact_shipments AS

SELECT
    s.shipment_id AS shipments_id,

    o.customer_id,
    o.store_id,
    s.order_id,

    CAST(s.status AS VARCHAR) AS status

FROM stg_shipments s

LEFT JOIN stg_orders o
    ON s.order_id = o.order_id
""")


# --------------------------------------------
# FACT PAYMENTS
# Grain: 1 Payment Transaction
# --------------------------------------------

con.execute("""
CREATE OR REPLACE TABLE fact_payments AS

SELECT
    p.payment_id,

    CAST(
        STRFTIME(
            TRY_CAST(o.order_date AS DATE),
            '%Y%m%d'
        ) AS INTEGER
    ) AS date_id,

    o.customer_id,
    o.store_id,
    p.order_id,

    CAST(p.amount AS NUMERIC) AS amount

FROM stg_payments p

LEFT JOIN stg_orders o
    ON p.order_id = o.order_id
""")

In [21]:
# ============================================
# COMPLETE DATA QUALITY CHECK
# ============================================

for table in tables:

    df = con.execute(
        f"SELECT * FROM stg_{table}"
    ).df()

    print("\n" + "=" * 80)
    print(f"TABLE: {table}")
    print("=" * 80)

    # ----------------------------------------
    # 1. ROW COUNT
    # ----------------------------------------
    print(f"\n[1] Row Count: {len(df):,}")

    # ----------------------------------------
    # 2. COLUMN COUNT
    # ----------------------------------------
    print(f"[2] Column Count: {len(df.columns)}")

    # ----------------------------------------
    # 3. MISSING / NULL
    # ----------------------------------------
    missing = df.isnull().sum()

    print("\n[3] Missing / NULL Values")

    if missing.sum() == 0:
        print("PASS - No Missing Values")
    else:
        print(
            pd.DataFrame({
                "missing_count": missing,
                "missing_percent": (
                    missing / len(df) * 100
                ).round(2)
            })[
                missing > 0
            ]
        )

    # ----------------------------------------
    # 4. DUPLICATE ROWS
    # ----------------------------------------
    duplicate_rows = df.duplicated().sum()

    print("\n[4] Duplicate Rows")
    print(f"Duplicate Rows = {duplicate_rows:,}")

    if duplicate_rows == 0:
        print("PASS")
    else:
        print("WARNING")

    # ----------------------------------------
    # 5. UNIQUE VALUES
    # ----------------------------------------
    print("\n[5] Unique Values")

    unique_check = pd.DataFrame({
        "column": df.columns,
        "unique_count": [
            df[col].nunique(dropna=True)
            for col in df.columns
        ],
        "total_rows": len(df)
    })

    unique_check["unique_percent"] = (
        unique_check["unique_count"]
        / unique_check["total_rows"]
        * 100
    ).round(2)

    print(unique_check.to_string(index=False))

    # ----------------------------------------
    # 6. PRIMARY KEY CHECK
    # ----------------------------------------
    if table in primary_keys:

        pk = primary_keys[table]

        print(f"\n[6] Primary Key Check: {pk}")

        if pk in df.columns:

            null_pk = df[pk].isnull().sum()

            duplicate_pk = (
                df[pk]
                .duplicated()
                .sum()
            )

            print(f"NULL PK       = {null_pk:,}")
            print(f"Duplicate PK  = {duplicate_pk:,}")

            if null_pk == 0 and duplicate_pk == 0:
                print("PASS - Primary Key is Unique and NOT NULL")
            else:
                print("FAIL - Primary Key Problem")

        else:
            print("FAIL - Primary Key Column Not Found")

    # ----------------------------------------
    # 7. DATA TYPES
    # ----------------------------------------
    print("\n[7] Data Types")
    print(df.dtypes)

    # ----------------------------------------
    # 8. NUMERIC NEGATIVE VALUES
    # ----------------------------------------
    numeric_cols = df.select_dtypes(
        include="number"
    ).columns

    if len(numeric_cols) > 0:

        print("\n[8] Negative Numeric Values")

        negative_check = []

        for col in numeric_cols:

            count = (
                df[col] < 0
            ).sum()

            if count > 0:
                negative_check.append({
                    "column": col,
                    "negative_count": count
                })

        if negative_check:
            print(
                pd.DataFrame(
                    negative_check
                ).to_string(index=False)
            )
        else:
            print("PASS - No Negative Values")

    # ----------------------------------------
    # 9. BLANK STRING CHECK
    # ----------------------------------------
    print("\n[9] Blank String Values")

    blank_values = {}

    for col in df.select_dtypes(
        include="object"
    ).columns:

        count = (
            df[col]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        )

        if count > 0:
            blank_values[col] = count

    if blank_values:
        print(
            pd.Series(
                blank_values,
                name="blank_count"
            )
        )
    else:
        print("PASS - No Blank String Values")


# ============================================
# OVERALL DATA QUALITY SUMMARY
# ============================================

print("\n" + "=" * 80)
print("OVERALL DATA QUALITY SUMMARY")
print("=" * 80)

summary = []

for table in tables:

    df = con.execute(
        f"SELECT * FROM stg_{table}"
    ).df()

    pk = primary_keys.get(table)

    null_count = int(
        df.isnull().sum().sum()
    )

    duplicate_rows = int(
        df.duplicated().sum()
    )

    if pk in df.columns:

        pk_null = int(
            df[pk].isnull().sum()
        )

        pk_duplicate = int(
            df[pk].duplicated().sum()
        )

    else:

        pk_null = None
        pk_duplicate = None

    numeric_cols = df.select_dtypes(
        include="number"
    ).columns

    negative_count = int(
        (df[numeric_cols] < 0)
        .sum()
        .sum()
    ) if len(numeric_cols) > 0 else 0

    blank_count = 0

    for col in df.select_dtypes(
        include="object"
    ).columns:

        blank_count += int(
            df[col]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        )

    if (
        null_count == 0
        and duplicate_rows == 0
        and (pk_null == 0 if pk_null is not None else True)
        and (pk_duplicate == 0 if pk_duplicate is not None else True)
        and negative_count == 0
        and blank_count == 0
    ):
        status = "PASS"
    else:
        status = "CHECK"

    summary.append({
        "table": table,
        "rows": len(df),
        "columns": len(df.columns),
        "null_count": null_count,
        "duplicate_rows": duplicate_rows,
        "pk_null": pk_null,
        "pk_duplicate": pk_duplicate,
        "negative_values": negative_count,
        "blank_strings": blank_count,
        "status": status
    })

summary_df = pd.DataFrame(summary)

print(
    summary_df.to_string(index=False)
)


TABLE: employees

[1] Row Count: 1,000
[2] Column Count: 3

[3] Missing / NULL Values
PASS - No Missing Values

[4] Duplicate Rows
Duplicate Rows = 0
PASS

[5] Unique Values
     column  unique_count  total_rows  unique_percent
employee_id          1000        1000           100.0
   store_id           100        1000            10.0
     salary           992        1000            99.2

[6] Primary Key Check: employee_id
NULL PK       = 0
Duplicate PK  = 0
PASS - Primary Key is Unique and NOT NULL

[7] Data Types
employee_id    int64
store_id       int64
salary         int64
dtype: object

[8] Negative Numeric Values
PASS - No Negative Values

[9] Blank String Values
PASS - No Blank String Values

TABLE: returns

[1] Row Count: 30,000
[2] Column Count: 3

[3] Missing / NULL Values
PASS - No Missing Values

[4] Duplicate Rows
Duplicate Rows = 0
PASS

[5] Unique Values
       column  unique_count  total_rows  unique_percent
    return_id         30000       30000          100.00
order_